# Benchmark de recuperación RAG — Manualito

Este cuaderno mide si el flujo real de recuperación de Manualito encuentra
el chunk que responde a cada pregunta del conjunto dorado. Está pensado
como anexo reproducible de la memoria del TFG: se puede leer sin
infraestructura y repetir la medición con el stack levantado.

## 1. Objetivo

Evaluar la recuperación con 24 preguntas sobre Catan y Monopoly. El
conjunto incluye preguntas que el buscador resuelve bien y los fallos
reales que destapó la auditoría. Así la métrica refleja tanto el caso
típico como el problemático. La línea base histórica es
`recall@5 = 0,7917`, `recall@10 = 0,7917` y `MRR = 0,4979`. Cualquier
mejora del buscador se compara contra estos números.

## 2. Metodología

Cada pregunta tiene un único chunk relevante anotado. Las métricas se
calculan con [`ranx`](https://amenra.github.io/ranx/), cuyas
implementaciones se contrastan en su propio proyecto con
[`trec_eval`](https://github.com/usnistgov/trec_eval), la referencia de
NIST. Como control local, el cuaderno recalcula cada métrica por
aritmética directa a partir de la posición del chunk esperado. Si ambos
valores discrepan, la ejecución aborta.

Los fallos de `recall@5` se muestran con la posición en la que quedó el
chunk. Un fallo de Docker o del servicio detiene la medición con
`[!] ERROR:`. Un problema de infraestructura no debe registrarse como un
fallo de relevancia.

## 3. Configuración

El modo predeterminado solo lee los resultados versionados y no necesita
Docker. Para medir se cambia `RUN_BENCHMARK` o se define
`MANUALITO_RAG_RUN=1`. `FORCE` sustituye una medición local ya registrada
con la misma configuración. Sin él, el cuaderno la reutiliza.

In [1]:
from __future__ import annotations

import hashlib
import importlib.metadata
import json
import math
import os
import subprocess
import sys
import shutil
import warnings
from collections.abc import Iterable, Iterator, Sequence
from contextlib import contextmanager
from datetime import UTC, datetime
from pathlib import Path
from typing import Any

from numba.core.errors import NumbaTypeSafetyWarning


RUN_BENCHMARK = False
FORCE = False

TOP_K = 20
METRICS = ("recall@5", "recall@10", "mrr")
FLOW_VERSION = 4
BENCHMARK_RELATIVE = Path("docs/benchmarks/rag/recuperacion")
CONTAINER_NAME = "manualito-rag"

YELLOW = "\033[33m"
CYAN = "\033[36m"
RED = "\033[31m"
RESET = "\033[0m"


def env_flag(name: str, default: bool) -> bool:
    value = os.getenv(name)
    if value is None:
        return default
    normalized = value.strip().lower()
    if normalized in {"1", "true", "yes", "on"}:
        return True
    if normalized in {"0", "false", "no", "off"}:
        return False
    raise ValueError(f"{name} debe ser 0/1, true/false, yes/no u on/off")


RUN_BENCHMARK = env_flag("MANUALITO_RAG_RUN", RUN_BENCHMARK)
FORCE = env_flag("MANUALITO_RAG_FORCE", FORCE)


def info(message: str) -> None:
    print(f"{YELLOW}[*]{RESET} {CYAN}{message}{RESET}")


def abort(message: str) -> None:
    print(f"{RED}[!] ERROR: {message}{RESET}", file=sys.stderr)
    raise RuntimeError(message)


def table(
    headers: Sequence[str],
    rows: Iterable[Sequence[object]],
    *,
    markdown: bool = False,
) -> str:
    rendered_rows = [[str(value) for value in row] for row in rows]
    rendered_headers = [str(header) for header in headers]
    if any(len(row) != len(rendered_headers) for row in rendered_rows):
        raise ValueError("todas las filas deben tener el mismo ancho")
    if markdown:
        lines = [
            "| " + " | ".join(rendered_headers) + " |",
            "| " + " | ".join("---" for _ in rendered_headers) + " |",
        ]
        lines.extend("| " + " | ".join(row) + " |" for row in rendered_rows)
        return "\n".join(lines)
    widths = [len(header) for header in rendered_headers]
    for row in rendered_rows:
        widths = [max(width, len(value)) for width, value in zip(widths, row, strict=True)]
    line = "-+-".join("-" * width for width in widths)
    output = [
        " | ".join(value.ljust(width) for value, width in zip(rendered_headers, widths, strict=True)),
        line,
    ]
    output.extend(
        " | ".join(value.ljust(width) for value, width in zip(row, widths, strict=True))
        for row in rendered_rows
    )
    return "\n".join(output)


def candidate_results_paths(start: Path) -> Iterator[Path]:
    resolved = start.resolve()
    for base in (resolved, *resolved.parents):
        yield base / "dataset" / "manifest.json"
        yield base / BENCHMARK_RELATIVE / "dataset" / "manifest.json"


def find_results_path(start: Path) -> Path:
    for candidate in candidate_results_paths(start):
        if candidate.is_file():
            return candidate.resolve()
    abort(
        "no se encontró dataset/manifest.json desde el directorio actual; "
        "ejecuta desde la raíz del repositorio o desde la carpeta del benchmark"
    )


def find_repo_root(benchmark_dir: Path) -> Path:
    for candidate in (benchmark_dir, *benchmark_dir.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    abort("no se encontró la raíz del repositorio a partir del benchmark")


def read_json(path: Path) -> Any:
    with path.open("r", encoding="utf-8") as stream:
        return json.load(stream)


def write_utf8(path: Path, text: str) -> None:
    with path.open("w", encoding="utf-8", newline="\n") as stream:
        stream.write(text)


@contextmanager
def ranx_runtime() -> Iterator[None]:
    previous = {
        name: os.environ.get(name)
        for name in ("IR_DATASETS_HOME", "MPLCONFIGDIR", "NUMBA_CACHE_DIR")
    }
    runtime = OUTPUT_DIR / ".ranx-runtime"
    if runtime.exists():
        shutil.rmtree(runtime)
    locations = {
        "IR_DATASETS_HOME": runtime / "ir_datasets",
        "MPLCONFIGDIR": runtime / "matplotlib",
        "NUMBA_CACHE_DIR": runtime / "numba",
    }
    runtime.mkdir()
    try:
        for name, location in locations.items():
            location.mkdir()
            os.environ[name] = str(location)
        with warnings.catch_warnings():
            # Avisos internos de ranx: cast de numba y escapes de su plantilla LaTeX.
            warnings.filterwarnings("ignore", category=NumbaTypeSafetyWarning)
            warnings.filterwarnings("ignore", message="invalid escape sequence", category=SyntaxWarning)
            yield
    finally:
        for name, value in previous.items():
            if value is None:
                os.environ.pop(name, None)
            else:
                os.environ[name] = value
        shutil.rmtree(runtime)


MANIFEST_ANCLA = find_results_path(Path.cwd())
BENCHMARK_DIR = MANIFEST_ANCLA.parent.parent
OUTPUT_DIR = BENCHMARK_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
RESULTS_PATH = OUTPUT_DIR / "results.json"
REPO_ROOT = find_repo_root(BENCHMARK_DIR)
DATASET_DIR = BENCHMARK_DIR / "dataset"
QUESTIONS_PATH = DATASET_DIR / "preguntas.json"
MANIFEST_PATH = DATASET_DIR / "manifest.json"

info(f"Benchmark localizado en {BENCHMARK_DIR}")
info(f"Modo: {'medición' if RUN_BENCHMARK else 'lectura'}; FORCE={FORCE}")

[*] Benchmark localizado en C:\Users\Ibai\Desktop\tfg github\manualito-feat-86\docs\benchmarks\rag\recuperacion
[*] Modo: medición; FORCE=True


## 4. Dataset versionado

El manifiesto conserva la procedencia de cada pregunta: origen, juego,
manual, chunk, modelo, semilla y fecha. Las 20 preguntas generadas fijan
la semilla con las opciones de la
[API de generación de Ollama](https://docs.ollama.com/api/generate). La
huella de integridad se calcula sobre el JSON parseado,
[serializado](https://docs.python.org/3.13/library/json.html) con claves
ordenadas, sin espacios y en UTF-8. Así los cambios de sangría o de
finales de línea no la invalidan.

In [2]:
def canonical_json_sha256(data: Any) -> str:
    # Es canónico porque ordena las claves y elimina espacios tras parsear el JSON.
    serialized = json.dumps(
        data,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    ).encode("utf-8")
    return hashlib.sha256(serialized).hexdigest()


def validate_dataset(questions: Any, manifest: Any) -> str:
    if not isinstance(questions, list) or len(questions) != 24:
        abort("preguntas.json debe contener exactamente 24 preguntas")
    required = {
        "id",
        "pregunta",
        "chunk_id",
        "game_id",
        "manual_id",
        "juego",
        "origen",
    }
    for index, question in enumerate(questions, start=1):
        if not isinstance(question, dict) or set(question) != required:
            abort(f"estructura no válida en la pregunta {index}")
        if any(not isinstance(question[field], str) or not question[field] for field in required):
            abort(f"campo vacío o no textual en la pregunta {index}")
    ids = [question["id"] for question in questions]
    if len(ids) != len(set(ids)):
        abort("los identificadores de pregunta no son únicos")
    if not isinstance(manifest, dict) or not isinstance(manifest.get("preguntas"), list):
        abort("manifest.json no contiene una lista de procedencias")
    provenance = manifest["preguntas"]
    if [entry.get("id") for entry in provenance] != ids:
        abort("manifest.json no está alineado con preguntas.json")
    for question, entry in zip(questions, provenance, strict=True):
        expected = {
            "origen": question["origen"],
            "juego": question["juego"],
            "juego_id": question["game_id"],
            "manual": question["manual_id"],
            "chunk_esperado": question["chunk_id"],
        }
        if any(entry.get(key) != value for key, value in expected.items()):
            abort(f"procedencia incoherente para {question['id']}")
    digest = canonical_json_sha256(questions)
    if manifest.get("sha256_canonico") != digest:
        abort("la huella canónica de preguntas.json no coincide con manifest.json")
    return digest


QUESTIONS = read_json(QUESTIONS_PATH)
MANIFEST = read_json(MANIFEST_PATH)
DATASET_SHA256 = validate_dataset(QUESTIONS, MANIFEST)
QUESTION_BY_ID = {question["id"]: question for question in QUESTIONS}

dataset_rows = []
for origin in ("gemma", "auditoria"):
    subset = [question for question in QUESTIONS if question["origen"] == origin]
    dataset_rows.append(
        (
            origin,
            len(subset),
            sum(question["juego"] == "Catan" for question in subset),
            sum(question["juego"] == "Monopoly" for question in subset),
        )
    )
print(table(("Origen", "Total", "Catan", "Monopoly"), dataset_rows))
info(f"Dataset verificado: {len(QUESTIONS)} preguntas; sha256={DATASET_SHA256}")


Origen    | Total | Catan | Monopoly
----------+-------+-------+---------
gemma     | 20    | 14    | 6       
auditoria | 4     | 1     | 3       
[*] Dataset verificado: 24 preguntas; sha256=36ddfefc3e42f02a7cde24f08393483ab8e47d59305deabf2952f992637a7d7d


## 5. Acceso al flujo real

El puerto del servicio RAG no se publica en el host. Por eso el cuaderno
envía un programa corto a `docker exec -i manualito-rag python -` y
consulta `/retrieve` desde dentro del contenedor. Desde la issue #84 el
filtro de permisos viaja en la propia consulta: la petición lleva los
manuales autorizados del dataset. Desde la issue #85 la pregunta de
búsqueda lleva delante "Manual de {juego}", como hace la API. Desde la
issue #86 la búsqueda es híbrida: la pregunta con prefijo alimenta la pata
semántica, la cruda alimenta la de palabras exactas y RRF fusiona sus
resultados. El programa solo replica la deduplicación por `content_hash`
que aplica el producto. Se usa el nombre fijo del contenedor porque el
proyecto que Docker Compose deduce del directorio cambia en los worktrees.

In [3]:
CONTAINER_SCRIPT = r'''
import json
import sys

import chromadb
import httpx


data = json.loads(INPUT_JSON)
questions = data["questions"]
authorized_by_game = data["authorized_by_game"]
top_k = data["top_k"]
collection = chromadb.HttpClient(host="chroma", port=8000).get_collection(
    "manualito_manuals"
)
results = []

with httpx.Client(
    base_url="http://127.0.0.1:8002",
    timeout=httpx.Timeout(60.0, connect=5.0),
) as client:
    for question in questions:
        response = client.post(
            "/retrieve",
            json={
                "lexical_question": question["pregunta"],
        "game_id": question["game_id"],
                "manual_ids": authorized_by_game[question["game_id"]],
                "question": f'Manual de {question["juego"]}: {question["pregunta"]}',
                "top_k": top_k,
            },
        )
        response.raise_for_status()
        chunks = response.json().get("chunks")
        if not isinstance(chunks, list):
            raise ValueError("/retrieve no devolvió una lista de chunks")
        raw_ids = []
        for chunk in chunks:
            chunk_id = chunk.get("id") if isinstance(chunk, dict) else None
            if not isinstance(chunk_id, str) or not chunk_id:
                raise ValueError("/retrieve devolvió un chunk sin ID válido")
            raw_ids.append(chunk_id)
        metadata_result = collection.get(
            ids=list(dict.fromkeys(raw_ids)),
            include=["metadatas"],
        ) if raw_ids else {"ids": [], "metadatas": []}
        metadata_by_id = dict(zip(
            metadata_result["ids"],
            metadata_result["metadatas"],
            strict=True,
        ))
        seen_hashes = set()
        ranking = []
        for chunk_id in raw_ids:
            metadata = metadata_by_id.get(chunk_id) or {}
            content_hash = metadata.get("content_hash")
            if not isinstance(content_hash, str) or not content_hash:
                continue
            if content_hash in seen_hashes:
                continue
            seen_hashes.add(content_hash)
            ranking.append(chunk_id)
        results.append({"id": question["id"], "ranking": ranking})

print("MANUALITO_BENCHMARK_JSON=" + json.dumps(results, ensure_ascii=False))
'''


def retrieve_real_flow(questions: list[dict[str, str]]) -> dict[str, list[str]]:
    authorized_by_game: dict[str, set[str]] = {}
    for question in questions:
        authorized_by_game.setdefault(question["game_id"], set()).add(question["manual_id"])
    payload = {
        "questions": questions,
        "authorized_by_game": {
            game: sorted(manuals) for game, manuals in authorized_by_game.items()
        },
        "top_k": TOP_K,
    }
    input_json = json.dumps(payload, ensure_ascii=False, separators=(",", ":"))
    program = f"INPUT_JSON = {input_json!r}\n" + CONTAINER_SCRIPT
    info(f"Consultando {len(questions)} preguntas dentro de {CONTAINER_NAME}")
    completed = subprocess.run(
        ["docker", "exec", "-i", CONTAINER_NAME, "python", "-"],
        input=program,
        capture_output=True,
        text=True,
        encoding="utf-8",
        timeout=30 * 60,
        check=False,
    )
    if completed.returncode != 0:
        detail = " ".join(completed.stderr.splitlines()).strip()
        abort(f"docker exec falló ({completed.returncode}): {detail or 'sin detalle'}")
    prefix = "MANUALITO_BENCHMARK_JSON="
    encoded = next(
        (line.removeprefix(prefix) for line in reversed(completed.stdout.splitlines()) if line.startswith(prefix)),
        None,
    )
    if encoded is None:
        abort("el ayudante del contenedor no devolvió resultados reconocibles")
    decoded = json.loads(encoded)
    if not isinstance(decoded, list):
        abort("el ayudante del contenedor devolvió una estructura no válida")
    rankings: dict[str, list[str]] = {}
    for result in decoded:
        question_id = result.get("id") if isinstance(result, dict) else None
        ranking = result.get("ranking") if isinstance(result, dict) else None
        if question_id not in QUESTION_BY_ID or not isinstance(ranking, list):
            abort("el ayudante devolvió un caso no válido")
        if len(ranking) > TOP_K or any(not isinstance(item, str) for item in ranking):
            abort(f"ranking no válido para {question_id}")
        rankings[question_id] = ranking
    if set(rankings) != set(QUESTION_BY_ID):
        abort("el ayudante no devolvió exactamente las preguntas del dataset")
    return rankings


info(f"Puente preparado hacia el contenedor {CONTAINER_NAME}")

[*] Puente preparado hacia el contenedor manualito-rag


## 6. Resultados

Cada medición local guarda la posición del chunk esperado y, como máximo,
los cinco primeros identificadores por pregunta. El ranking completo de
20 solo existe en memoria durante la ejecución. Las salidas son
idempotentes: una configuración ya medida se reutiliza salvo que se
fuerce el recálculo.

In [4]:
def expected_position(ranking: Sequence[str], expected_chunk: str) -> int | None:
    try:
        return ranking.index(expected_chunk) + 1
    except ValueError:
        return None


def direct_metrics(
    question_ids: Sequence[str],
    rankings: dict[str, list[str]],
) -> dict[str, float]:
    positions = [
        expected_position(rankings[question_id], QUESTION_BY_ID[question_id]["chunk_id"])
        for question_id in question_ids
    ]
    total = len(positions)
    return {
        "recall@5": sum(position is not None and position <= 5 for position in positions) / total,
        "recall@10": sum(position is not None and position <= 10 for position in positions) / total,
        "mrr": sum(1.0 / position if position is not None else 0.0 for position in positions) / total,
    }


def ranx_metrics(
    question_ids: Sequence[str],
    rankings: dict[str, list[str]],
) -> dict[str, float]:
    from ranx import Qrels, Run, evaluate

    qrels = Qrels(
        {
            question_id: {QUESTION_BY_ID[question_id]["chunk_id"]: 1}
            for question_id in question_ids
        }
    )
    run = Run(
        {
            question_id: {
                chunk_id: float(len(rankings[question_id]) - index)
                for index, chunk_id in enumerate(rankings[question_id])
            }
            for question_id in question_ids
        }
    )
    evaluated = evaluate(qrels, run, list(METRICS))
    scores = {metric: float(evaluated[metric]) for metric in METRICS}
    direct = direct_metrics(question_ids, rankings)
    for metric in METRICS:
        if not math.isclose(scores[metric], direct[metric], rel_tol=1e-6, abs_tol=1e-6):
            abort(f"ranx y el control aritmético discrepan en {metric}")
    return scores


def metrics_for_cases(
    question_ids: Sequence[str],
    rankings: dict[str, list[str]],
) -> dict[str, int | float]:
    return {
        "total": len(question_ids),
        "errores": 0,
        **ranx_metrics(question_ids, rankings),
    }


def grouped_metrics(
    field: str,
    rankings: dict[str, list[str]],
) -> dict[str, dict[str, int | float]]:
    groups = sorted({question[field] for question in QUESTIONS})
    return {
        group: metrics_for_cases(
            [question["id"] for question in QUESTIONS if question[field] == group],
            rankings,
        )
        for group in groups
    }


CONFIGURATION = {
    "dataset_sha256_canonico": DATASET_SHA256,
    "flujo_version": FLOW_VERSION,
    "top_k_recuperacion": TOP_K,
    "metricas": list(METRICS),
    "contenedor": CONTAINER_NAME,
    "postprocesado": "permisos filtrados en la consulta y deduplicación por content_hash",
}
CONFIGURATION_HASH = canonical_json_sha256(CONFIGURATION)


def git_revision() -> str | None:
    completed = subprocess.run(
        ["git", "rev-parse", "HEAD"],
        cwd=REPO_ROOT,
        capture_output=True,
        text=True,
        encoding="utf-8",
        check=False,
    )
    return completed.stdout.strip() or None if completed.returncode == 0 else None


def build_measurement(rankings: dict[str, list[str]]) -> dict[str, Any]:
    cases = []
    for question in QUESTIONS:
        ranking = rankings[question["id"]]
        cases.append(
            {
                "id": question["id"],
                "posicion": expected_position(ranking, question["chunk_id"]),
                "top_5": ranking[:5],
            }
        )
    return {
        "id": f"local-{CONFIGURATION_HASH[:12]}",
        "tipo": "medicion_local",
        "fecha": datetime.now(UTC).isoformat(timespec="seconds").replace("+00:00", "Z"),
        "dataset_sha256_canonico": DATASET_SHA256,
        "configuracion_hash": CONFIGURATION_HASH,
        "revision_git": git_revision(),
        "ranx_version": importlib.metadata.version("ranx"),
        "evaluador": "ranx; métricas verificadas por ranx contra trec_eval",
        "top_5_disponibles": True,
        "configuracion": CONFIGURATION,
        "metricas": metrics_for_cases(list(QUESTION_BY_ID), rankings),
        "por_juego": grouped_metrics("juego", rankings),
        "por_origen": grouped_metrics("origen", rankings),
        "casos": cases,
    }


def validate_results(bundle: Any) -> dict[str, Any]:
    if not isinstance(bundle, dict) or bundle.get("version") != 2:
        abort("outputs/results.json no cumple la versión 2 del contrato")
    measurements = bundle.get("mediciones")
    if not isinstance(measurements, list) or not measurements:
        abort("outputs/results.json no contiene mediciones")
    if measurements[0].get("tipo") != "linea_base_auditoria":
        abort("la primera medición debe ser la línea base histórica")
    expected_ids = list(QUESTION_BY_ID)
    measurement_ids: set[str] = set()
    for measurement in measurements:
        if not isinstance(measurement, dict) or not isinstance(measurement.get("id"), str):
            abort("outputs/results.json contiene una medición no válida")
        if measurement["id"] in measurement_ids:
            abort(f"identificador de medición repetido: {measurement['id']}")
        measurement_ids.add(measurement["id"])
        if measurement.get("dataset_sha256_canonico") != DATASET_SHA256:
            abort(f"dataset distinto en la medición {measurement['id']}")
        metrics = measurement.get("metricas")
        if not isinstance(metrics, dict) or metrics.get("total") != len(expected_ids):
            abort(f"métricas incompletas en {measurement['id']}")
        for metric in METRICS:
            if not isinstance(metrics.get(metric), (int, float)):
                abort(f"falta {metric} en {measurement['id']}")
        available = measurement.get("top_5_disponibles")
        cases = measurement.get("casos")
        if not isinstance(available, bool) or not isinstance(cases, list):
            abort(f"casos no válidos en {measurement['id']}")
        if [case.get("id") for case in cases if isinstance(case, dict)] != expected_ids:
            abort(f"casos desalineados en {measurement['id']}")
        for case in cases:
            if set(case) != {"id", "posicion", "top_5"}:
                abort(f"caso no compacto en {measurement['id']}")
            position = case["posicion"]
            if position is not None and (
                isinstance(position, bool) or not isinstance(position, int) or not 1 <= position <= TOP_K
            ):
                abort(f"posición no válida para {case['id']}")
            top_five = case["top_5"]
            if available:
                if not isinstance(top_five, list) or len(top_five) > 5:
                    abort(f"top_5 no válido para {case['id']}")
                if any(not isinstance(chunk_id, str) for chunk_id in top_five):
                    abort(f"candidato no válido para {case['id']}")
            elif top_five is not None:
                abort(f"la línea histórica no debe inventar candidatos para {case['id']}")
    return bundle


def load_results() -> dict[str, Any]:
    return validate_results(read_json(RESULTS_PATH))


def format_metric(value: object) -> str:
    return f"{float(value):.4f}".replace(".", ",")


def failed_cases(measurement: dict[str, Any]) -> list[dict[str, Any]]:
    return [
        case
        for case in measurement["casos"]
        if case["posicion"] is None or case["posicion"] > 5
    ]


def render_results_markdown(bundle: dict[str, Any]) -> str:
    summary_rows = []
    for measurement in bundle["mediciones"]:
        metrics = measurement["metricas"]
        summary_rows.append(
            (
                measurement["id"],
                measurement["fecha"],
                metrics["total"],
                format_metric(metrics["recall@5"]),
                format_metric(metrics["recall@10"]),
                format_metric(metrics["mrr"]),
                metrics["errores"],
            )
        )
    latest = bundle["mediciones"][-1]
    lines = [
        "# Resultados de recuperación RAG",
        "",
        table(
            ("Medición", "Fecha", "Preguntas", "Recall@5", "Recall@10", "MRR", "Errores"),
            summary_rows,
            markdown=True,
        ),
        "",
        f"## Desglose de `{latest['id']}`",
        "",
    ]
    for title, field in (("Por juego", "por_juego"), ("Por origen", "por_origen")):
        rows = [
            (
                name,
                metrics["total"],
                format_metric(metrics["recall@5"]),
                format_metric(metrics["recall@10"]),
                format_metric(metrics["mrr"]),
            )
            for name, metrics in latest[field].items()
        ]
        lines.extend(
            [
                f"### {title}",
                "",
                table(("Grupo", "Total", "Recall@5", "Recall@10", "MRR"), rows, markdown=True),
                "",
            ]
        )
    failure_rows = []
    for case in failed_cases(latest):
        question = QUESTION_BY_ID[case["id"]]
        failure_rows.append(
            (
                case["id"],
                question["juego"],
                question["origen"],
                case["posicion"] if case["posicion"] is not None else f">{TOP_K}",
                question["pregunta"],
            )
        )
    lines.extend(
        [
            "## Casos fallidos en recall@5",
            "",
            table(("ID", "Juego", "Origen", "Posición", "Pregunta"), failure_rows, markdown=True),
            "",
            "La línea histórica no conserva candidatos (`top_5 = null`). Las mediciones locales guardan como máximo cinco identificadores por pregunta.",
            "",
        ]
    )
    return "\n".join(lines)


def render_conclusions_markdown(bundle: dict[str, Any]) -> str:
    latest = bundle["mediciones"][-1]
    metrics = latest["metricas"]
    failures = failed_cases(latest)
    successes = int(round(float(metrics["recall@5"]) * int(metrics["total"])))
    lines = [
        "# Conclusiones de recuperación RAG",
        "",
        f"- `{latest['id']}` recupera el chunk esperado entre los cinco primeros en {successes} de {metrics['total']} preguntas: **recall@5 = {format_metric(metrics['recall@5'])}**.",
        f"- El recall no cambia al ampliar de 5 a 10 resultados: **recall@10 = {format_metric(metrics['recall@10'])}**.",
        f"- La posición media recíproca es **MRR = {format_metric(metrics['mrr'])}**.",
        f"- Hay {len(failures)} fallos de recall@5; su posición queda registrada en `results.json` y resumida en `results.md`.",
        "- El benchmark evalúa recuperación, no la fidelidad de la respuesta generada por el LLM.",
        "",
    ]
    return "\n".join(lines)


def save_results(bundle: dict[str, Any]) -> None:
    validate_results(bundle)
    write_utf8(RESULTS_PATH, json.dumps(bundle, ensure_ascii=False, indent=2) + "\n")
    write_utf8(OUTPUT_DIR / "results.md", render_results_markdown(bundle))
    write_utf8(OUTPUT_DIR / "conclusions.md", render_conclusions_markdown(bundle))


def print_report(measurement: dict[str, Any]) -> None:
    metrics = measurement["metricas"]
    print("Resumen de métricas")
    print(
        table(
            ("Medición", "Preguntas", "Recall@5", "Recall@10", "MRR", "Errores"),
            [
                (
                    measurement["id"],
                    metrics["total"],
                    format_metric(metrics["recall@5"]),
                    format_metric(metrics["recall@10"]),
                    format_metric(metrics["mrr"]),
                    metrics["errores"],
                )
            ],
        )
    )
    print()
    print("Casos fallidos en recall@5")
    rows = []
    for case in failed_cases(measurement):
        question = QUESTION_BY_ID[case["id"]]
        rows.append(
            (
                case["id"],
                question["juego"],
                question["origen"],
                case["posicion"] if case["posicion"] is not None else f">{TOP_K}",
                question["pregunta"],
            )
        )
    print(table(("ID", "Juego", "Origen", "Posición", "Pregunta"), rows))


info("Contrato de resultados y tablas del informe preparados")

[*] Contrato de resultados y tablas del informe preparados


## 7. Conclusiones

La última celda muestra siempre las tablas. En modo lectura carga la
última entrada persistida sin consultar Docker. En modo medición
reutiliza la entrada compatible o la recalcula con `FORCE = True`.

In [5]:
results_bundle = load_results()

if RUN_BENCHMARK:
    compatible_index = next(
        (
            index
            for index, measurement in enumerate(results_bundle["mediciones"])
            if measurement.get("tipo") == "medicion_local"
            and measurement.get("configuracion_hash") == CONFIGURATION_HASH
        ),
        None,
    )
    if compatible_index is not None and not FORCE:
        selected_measurement = results_bundle["mediciones"][compatible_index]
        info(f"Se reutiliza la medición compatible {selected_measurement['id']}")
    else:
        measured_rankings = retrieve_real_flow(QUESTIONS)
        with ranx_runtime():
            selected_measurement = build_measurement(measured_rankings)
        if compatible_index is None:
            results_bundle["mediciones"].append(selected_measurement)
        else:
            results_bundle["mediciones"][compatible_index] = selected_measurement
        save_results(results_bundle)
        info(f"Medición persistida en {RESULTS_PATH}")
else:
    selected_measurement = results_bundle["mediciones"][-1]
    info(
        "Modo lectura: no se consulta Docker; se muestran los resultados "
        f"persistidos de {selected_measurement['id']}"
    )

print_report(selected_measurement)


[*] Consultando 24 preguntas dentro de manualito-rag


[*] Medición persistida en C:\Users\Ibai\Desktop\tfg github\manualito-feat-86\docs\benchmarks\rag\recuperacion\outputs\results.json
Resumen de métricas
Medición           | Preguntas | Recall@5 | Recall@10 | MRR    | Errores
-------------------+-----------+----------+-----------+--------+--------
local-957613c07305 | 24        | 0,9167   | 1,0000    | 0,7125 | 0      

Casos fallidos en recall@5
ID           | Juego    | Origen | Posición | Pregunta                                  
-------------+----------+--------+----------+-------------------------------------------
pregunta-004 | Catan    | gemma  | 10       | ¿Qué debe recibir cada jugador al empezar?
pregunta-020 | Monopoly | gemma  | 6        | ¿Cómo se empieza a jugar?                 
